In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/alokmanawat/loraweights/lora_weights_only.pth
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch
import torch.nn as nn 
from torch.utils.data import Dataset , DataLoader
import os 
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback ,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from torch.optim import AdamW
import string 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pickle

In [3]:
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

In [4]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

import wandb
wandb.login()  

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 23f2001025 (23f2001025-indian-institue-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
sam = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [6]:
sam

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A B C
3,4,A B C
4,5,A B C
...,...,...
495,496,A B C
496,497,A B C
497,498,A B C
498,499,A B C


In [7]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [8]:
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [9]:
sam = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [10]:
import warnings
warnings.filterwarnings('ignore')

In [11]:
LABEL_2_IDX = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
IDX_2_LABEL = {v: k for k, v in LABEL_2_IDX.items()}
OPTIONS = ['A', 'B', 'C', 'D', 'E']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [12]:
train_df = train.copy()

In [13]:
train_df['answer'] = train_df['answer'].apply(lambda x:LABEL_2_IDX[x])

In [14]:
train_df

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,0
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,1
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,4
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,3
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,1


In [15]:
train_set,val_set = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['answer'])

In [16]:
train_set = train_set.reset_index(drop=True)
val_set = val_set.reset_index(drop=True)

In [17]:
""""MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN = 256  

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded:', MODEL_NAME)"""

'"MODEL_NAME = \'microsoft/deberta-v3-base\'\nMAX_LEN = 256  \n\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\nprint(\'Tokenizer loaded:\', MODEL_NAME)'

In [18]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        choices = [str(row[opt]) for opt in OPTIONS]

        
        encodings = self.tokenizer(
            [prompt] * 5,
            choices,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        item = {
            'input_ids':      encodings['input_ids'],       
            'attention_mask': encodings['attention_mask'], 
        }
        if 'token_type_ids' in encodings:
            item['token_type_ids'] = encodings['token_type_ids']

        if not self.is_test:
            item['labels'] = torch.tensor(row['answer'], dtype=torch.long)

        return item

In [19]:
#train_data = MCQDataset(train_set,tokenizer)
#val_data = MCQDataset(val_set,tokenizer)


In [20]:
wt_path = "/kaggle/input/datasets/alokmanawat/loraweights/lora_weights_only.pth"

In [21]:
"""lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                            # LoRA rank — increase for more capacity
    lora_alpha=32,                   # Scaling factor
    lora_dropout=0.1,
    target_modules=['query_proj', 'key_proj', 'value_proj'],  # DeBERTa attention layers
    bias='none'
)

base_model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True
)



model = get_peft_model(base_model, lora_config)


base_model = AutoModelForMultipleChoice.from_pretrained('microsoft/deberta-v3-base')
model = get_peft_model(base_model, lora_config)

model.load_state_dict(
    torch.load(wt_path),
    strict=False
)
model = model.to(device)

"""


"lora_config = LoraConfig(\n    task_type=TaskType.SEQ_CLS,\n    r=16,                            # LoRA rank — increase for more capacity\n    lora_alpha=32,                   # Scaling factor\n    lora_dropout=0.1,\n    target_modules=['query_proj', 'key_proj', 'value_proj'],  # DeBERTa attention layers\n    bias='none'\n)\n\nbase_model = AutoModelForMultipleChoice.from_pretrained(\n    MODEL_NAME,\n    ignore_mismatched_sizes=True\n)\n\n\n\nmodel = get_peft_model(base_model, lora_config)\n\n\nbase_model = AutoModelForMultipleChoice.from_pretrained('microsoft/deberta-v3-base')\nmodel = get_peft_model(base_model, lora_config)\n\nmodel.load_state_dict(\n    torch.load(wt_path),\n    strict=False\n)\nmodel = model.to(device)\n\n"

In [22]:
def map_at_3(predictions, labels):
    map_score = 0.0
    for pred_top3, true_label in zip(predictions, labels):
        score = 0.0
        num_hits = 0
        for k, p in enumerate(pred_top3[:3], 1):
            if p == true_label:
                num_hits += 1
                score += num_hits / k
        map_score += score
    return map_score / len(labels)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # logits shape: (batch, 5)
    top3_preds = np.argsort(logits, axis=-1)[:, ::-1][:, :3]  # top 3 indices
    
    # Accuracy (top-1)
    top1_acc = (top3_preds[:, 0] == labels).mean()
    
    # MAP@3
    map3 = map_at_3(top3_preds, labels)
    
    return {'accuracy': top1_acc, 'map@3': map3}

In [23]:
""""training_args = TrainingArguments(
    output_dir='./deberta-mcq-lora',
    num_train_epochs=5,
    per_device_train_batch_size=4,      
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,    
    warmup_ratio=0.1,
    learning_rate=2e-4,                
    weight_decay=0.01,
    fp16=False,
    bf16=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='map@3',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',                  
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Starting training...')
trainer.train()"""

'"training_args = TrainingArguments(\n    output_dir=\'./deberta-mcq-lora\',\n    num_train_epochs=5,\n    per_device_train_batch_size=4,      \n    per_device_eval_batch_size=8,\n    gradient_accumulation_steps=4,    \n    warmup_ratio=0.1,\n    learning_rate=2e-4,                \n    weight_decay=0.01,\n    fp16=False,\n    bf16=False,\n    eval_strategy=\'epoch\',\n    save_strategy=\'epoch\',\n    load_best_model_at_end=True,\n    metric_for_best_model=\'map@3\',\n    greater_is_better=True,\n    logging_steps=50,\n    report_to=\'none\',                  \n    seed=42\n)\n\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_data,\n    eval_dataset=val_data,\n    processing_class=tokenizer,\n    compute_metrics=compute_metrics,\n    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]\n)\n\nprint(\'Starting training...\')\ntrainer.train()'

In [24]:
""""torch.save(
    {k: v for k, v in model.state_dict().items() if 'lora' in k},
    '/kaggle/working/lora_weights_only.pth'
)"""

'"torch.save(\n    {k: v for k, v in model.state_dict().items() if \'lora\' in k},\n    \'/kaggle/working/lora_weights_only.pth\'\n)'

In [25]:
# Evaluate on validation set
#results = trainer.evaluate()
#print('\nValidation Results:')
#for k, v in results.items():
#    print(f'  {k}: {v:.4f}')

In [26]:
def predict_top3(model, tokenizer, df, max_len=256, batch_size=8):
    """Run inference and return top-3 predicted labels for each row."""
    model.eval()
    dataset = MCQDataset(df, tokenizer, max_len=max_len, is_test=True)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_logits = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)

            outputs = model(**kwargs)
            all_logits.append(outputs.logits.cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)  # (N, 5)

    # Rank options by logit score (descending)
    top3_indices = np.argsort(all_logits, axis=-1)[:, ::-1][:, :3]
    top3_labels  = [[IDX_2_LABEL[i] for i in row] for row in top3_indices]
    predictions  = [' '.join(labels) for labels in top3_labels]

    return predictions


#print('Running inference on test set...')
#test_preds = predict_top3(model, tokenizer, test, max_len=MAX_LEN)

In [27]:
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 192
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 15
LR = 3e-5
SAVE_PATH = "deberta_base_lora.pt"


In [28]:
"""os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
QUESTION_COL = "prompt2"
ANSWER_COL = "answer"

df = pd.read_csv(TRAIN_PATH)
df.columns = [c.strip() for c in df.columns]
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL_MAP = {v: i for i, v in enumerate(OPTION_COLS)}

df["label_idx"] = df[ANSWER_COL].str.strip().map(LABEL_MAP)
df = df.dropna(subset=["label_idx"])
df["label_idx"] = df["label_idx"].astype(int)
df["prompt2"] = df["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))

train_df, val_df = train_test_split(df, test_size=0.1, random_state=7, stratify=df["label_idx"])


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)"""

'os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"\nQUESTION_COL = "prompt2"\nANSWER_COL = "answer"\n\ndf = pd.read_csv(TRAIN_PATH)\ndf.columns = [c.strip() for c in df.columns]\nprint("Columns:", df.columns.tolist())\nprint("Shape:", df.shape)\n\nOPTION_COLS = ["A", "B", "C", "D", "E"]\nLABEL_MAP = {v: i for i, v in enumerate(OPTION_COLS)}\n\ndf["label_idx"] = df[ANSWER_COL].str.strip().map(LABEL_MAP)\ndf = df.dropna(subset=["label_idx"])\ndf["label_idx"] = df["label_idx"].astype(int)\ndf["prompt2"] = df["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))\n\ntrain_df, val_df = train_test_split(df, test_size=0.1, random_state=7, stratify=df["label_idx"])\n\n\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)'

In [29]:
def shuffle_options(row):
    options = {col: row[col] for col in OPTION_COLS}
    answer_text = options[row[ANSWER_COL].strip()]
    keys = OPTION_COLS.copy()
    np.random.shuffle(keys)
    new_row = row.copy()
    for i, col in enumerate(OPTION_COLS):
        new_row[col] = options[keys[i]]
    for col in OPTION_COLS:
        if new_row[col] == answer_text:
            new_row[ANSWER_COL] = col
            break
    new_row["label_idx"] = LABEL_MAP[new_row[ANSWER_COL]]
    return new_row

In [30]:
class MCQDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        question = str(row[QUESTION_COL])
        input_ids_list, attn_mask_list = [], []
        for col in OPTION_COLS:
            enc = self.tokenizer(
                question,
                str(row[col]),
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids_list.append(enc["input_ids"].squeeze(0))
            attn_mask_list.append(enc["attention_mask"].squeeze(0))
        label = torch.tensor(row["label_idx"], dtype=torch.long)
        return {
            "input_ids": torch.stack(input_ids_list),
            "attention_mask": torch.stack(attn_mask_list),
            "labels": label
        }

In [31]:
"""lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["query_proj", "key_proj", "value_proj", "out_proj"],
    bias="none"
)

base_model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
)

base_model.gradient_checkpointing_enable()

model = get_peft_model(base_model, lora_config)

model.load_state_dict(
    torch.load(wt_path),
    strict=False
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float
        

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01
)
"""

'lora_config = LoraConfig(\n    task_type=TaskType.SEQ_CLS,\n    r=64,\n    lora_alpha=128,\n    lora_dropout=0.05,\n    target_modules=["query_proj", "key_proj", "value_proj", "out_proj"],\n    bias="none"\n)\n\nbase_model = AutoModelForMultipleChoice.from_pretrained(\n    MODEL_NAME,\n    ignore_mismatched_sizes=True,\n    torch_dtype=torch.float32\n)\n\nbase_model.gradient_checkpointing_enable()\n\nmodel = get_peft_model(base_model, lora_config)\n\nmodel.load_state_dict(\n    torch.load(wt_path),\n    strict=False\n)\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nmodel = model.to(device)\n\nfor name, param in model.named_parameters():\n    if param.requires_grad:\n        param.data = param.data.float\n        \n\noptimizer = AdamW(\n    [p for p in model.parameters() if p.requires_grad],\n    lr=LR,\n    weight_decay=0.01\n)\n'

In [32]:
"""train_ds = MCQDataset(train_df, tokenizer, MAX_LEN)
val_ds = MCQDataset(val_df, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)"""


'train_ds = MCQDataset(train_df, tokenizer, MAX_LEN)\nval_ds = MCQDataset(val_df, tokenizer, MAX_LEN)\ntrain_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)\nval_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)'

In [33]:
"""steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, total_steps // 10)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)"""

'steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)\ntotal_steps = steps_per_epoch * EPOCHS\nwarmup_steps = max(1, total_steps // 10)\nscheduler = get_cosine_schedule_with_warmup(\n    optimizer,\n    num_warmup_steps=warmup_steps,\n    num_training_steps=total_steps\n)'

In [34]:
def map3_loss(logits, labels):
    ce = F.cross_entropy(logits.float(), labels)
    top3 = torch.topk(logits, k=3, dim=-1).indices
    in_top3 = (top3 == labels.unsqueeze(1)).any(dim=1).float()
    recall_penalty = (1 - in_top3).mean()
    return ce + 0.5 * recall_penalty

def compute_map3(logits_all, labels_all):
    scores = []
    for logits, label in zip(logits_all, labels_all):
        top3 = np.argsort(logits)[::-1][:3].tolist()
        score = 0.0
        for rank, pred in enumerate(top3):
            if pred == int(label):
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return float(np.mean(scores))

In [35]:
"""run = wandb.init(
    project="23f2001025-t22026",       
    entity="23f2001025-indian-institue-of-technology-madras",  
    name="deberta-v3-base",       
    config={
        "model": "deberta-v3-base",
        "Optim":"AdamW",
        "epochs": 17,
        "batch_size": 2,
        "lr": 3e-6
    }
)
best_map3 = 0.0
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.float()
        loss = map3_loss(logits, labels) / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    if len(train_loader) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.extend(outputs.logits.float().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_map3 = compute_map3(all_logits, all_labels)
    avg_loss = total_loss / max(1, len(train_loader))
    wandb.log({
       "avg_loss":avg_loss,
       "val_map3":val_map3,
       
    })
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val MAP@3: {val_map3:.4f}")
    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save(model.state_dict(), "deberta_base_lora7.pt")
        print(f"  Saved best model with MAP@3={best_map3:.4f}")
    

"""

'run = wandb.init(\n    project="23f2001025-t22026",       \n    entity="23f2001025-indian-institue-of-technology-madras",  \n    name="deberta-v3-base",       \n    config={\n        "model": "deberta-v3-base",\n        "Optim":"AdamW",\n        "epochs": 17,\n        "batch_size": 2,\n        "lr": 3e-6\n    }\n)\nbest_map3 = 0.0\nfor epoch in range(EPOCHS):\n    model.train()\n    optimizer.zero_grad()\n    total_loss = 0.0\n    for step, batch in enumerate(train_loader):\n        input_ids = batch["input_ids"].to(device)\n        attention_mask = batch["attention_mask"].to(device)\n        labels = batch["labels"].to(device)\n        outputs = model(input_ids=input_ids, attention_mask=attention_mask)\n        logits = outputs.logits.float()\n        loss = map3_loss(logits, labels) / GRAD_ACCUM\n        loss.backward()\n        total_loss += loss.item() * GRAD_ACCUM\n        if (step + 1) % GRAD_ACCUM == 0:\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n     

In [36]:
test_df = pd.read_csv(TEST_PATH)

In [37]:
test_df

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


In [38]:
class MCQTestDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        question = str(row[QUESTION_COL])
        input_ids_list, attn_mask_list = [], []
        for col in OPTION_COLS:
            enc = self.tokenizer(
                question,
                str(row[col]),
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids_list.append(enc["input_ids"].squeeze(0))
            attn_mask_list.append(enc["attention_mask"].squeeze(0))
        return {
            "input_ids": torch.stack(input_ids_list),
            "attention_mask": torch.stack(attn_mask_list),
        }





In [39]:
"""test_df = pd.read_csv(TEST_PATH)
test_df.columns = [c.strip() for c in test_df.columns]
test_df["prompt2"] = test_df["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))"""

'test_df = pd.read_csv(TEST_PATH)\ntest_df.columns = [c.strip() for c in test_df.columns]\ntest_df["prompt2"] = test_df["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))'

In [40]:
""""model = get_peft_model(base_model, lora_config)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device), strict=False)
model = model.to(device)
model.eval()"""

'"model = get_peft_model(base_model, lora_config)\nmodel.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device), strict=False)\nmodel = model.to(device)\nmodel.eval()'

In [41]:
"""test_ds = MCQTestDataset(test_df, tokenizer, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)"""


'test_ds = MCQTestDataset(test_df, tokenizer, MAX_LEN)\ntest_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)'

In [42]:
"""all_probs = []
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        all_probs.extend(probs)
        if (i + 1) % 20 == 0:
            print(f"  Processed {(i+1)*BATCH_SIZE}/{len(test_df)} rows")

predictions = []
for probs in all_probs:
    top3_idx = np.argsort(probs)[::-1][:3]
    top3_labels = " ".join([OPTION_COLS[i] for i in top3_idx])
    predictions.append(top3_labels)"""


'all_probs = []\nwith torch.no_grad():\n    for i, batch in enumerate(test_loader):\n        input_ids = batch["input_ids"].to(device)\n        attention_mask = batch["attention_mask"].to(device)\n        outputs = model(input_ids=input_ids, attention_mask=attention_mask)\n        probs = F.softmax(outputs.logits.float(), dim=-1).cpu().numpy()\n        all_probs.extend(probs)\n        if (i + 1) % 20 == 0:\n            print(f"  Processed {(i+1)*BATCH_SIZE}/{len(test_df)} rows")\n\npredictions = []\nfor probs in all_probs:\n    top3_idx = np.argsort(probs)[::-1][:3]\n    top3_labels = " ".join([OPTION_COLS[i] for i in top3_idx])\n    predictions.append(top3_labels)'

In [43]:
"""id_col = test_df["id"]

submission = pd.DataFrame({"ID": id_col, "Prediction": predictions})
submission.to_csv("submission.csv", index=False)

print(f"Total predictions: {len(submission)}")
print(submission.head(10))"""

'id_col = test_df["id"]\n\nsubmission = pd.DataFrame({"ID": id_col, "Prediction": predictions})\nsubmission.to_csv("submission.csv", index=False)\n\nprint(f"Total predictions: {len(submission)}")\nprint(submission.head(10))'

In [44]:

"""os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 192
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 17
LR = 2e-5
SAVE_PATH = "deberta_base_lora.pt"
QUESTION_COL = "prompt2"
ANSWER_COL = "answer"
PATIENCE = 4

df = pd.read_csv(TRAIN_PATH)
df["prompt2"] = df["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))
df.columns = [c.strip() for c in df.columns]
test_df_check = pd.read_csv(TEST_PATH)
test_df_check["prompt2"] = test_df_check["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))
test_df_check.columns = [c.strip() for c in test_df_check.columns]


OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL_MAP = {v: i for i, v in enumerate(OPTION_COLS)}

df["label_idx"] = df[ANSWER_COL].str.strip().map(LABEL_MAP)
df = df.dropna(subset=["label_idx"])
df["label_idx"] = df["label_idx"].astype(int)


train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df["label_idx"])


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def shuffle_options(row):
    options = {col: row[col] for col in OPTION_COLS}
    answer_text = options[row[ANSWER_COL].strip()]
    keys = OPTION_COLS.copy()
    np.random.shuffle(keys)
    new_row = row.copy()
    for i, col in enumerate(OPTION_COLS):
        new_row[col] = options[keys[i]]
    for col in OPTION_COLS:
        if new_row[col] == answer_text:
            new_row[ANSWER_COL] = col
            break
    new_row["label_idx"] = LABEL_MAP[new_row[ANSWER_COL]]
    return new_row

class MCQDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, augment=False):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx].copy()
        if self.augment and np.random.random() < 0.5:
            row = shuffle_options(row)
        question = str(row[QUESTION_COL])
        input_ids_list, attn_mask_list = [], []
        for col in OPTION_COLS:
            enc = self.tokenizer(
                question,
                str(row[col]),
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids_list.append(enc["input_ids"].squeeze(0))
            attn_mask_list.append(enc["attention_mask"].squeeze(0))
        label = torch.tensor(row["label_idx"], dtype=torch.long)
        return {
            "input_ids": torch.stack(input_ids_list),
            "attention_mask": torch.stack(attn_mask_list),
            "labels": label
        }

train_ds = MCQDataset(train_df, tokenizer, MAX_LEN, augment=True)
val_ds = MCQDataset(val_df, tokenizer, MAX_LEN, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.15,
    target_modules=["query_proj", "key_proj", "value_proj", "out_proj"],
    bias="none"
)

base_model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
)
base_model.gradient_checkpointing_enable()
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.05
)

steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, total_steps // 8)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

def map3_loss(logits, labels):
    ce = F.cross_entropy(logits.float(), labels, label_smoothing=0.1)
    top3 = torch.topk(logits, k=3, dim=-1).indices
    in_top3 = (top3 == labels.unsqueeze(1)).any(dim=1).float()
    recall_penalty = (1 - in_top3).mean()
    return ce + 0.5 * recall_penalty

def compute_map3(logits_all, labels_all):
    scores = []
    for logits, label in zip(logits_all, labels_all):
        top3 = np.argsort(logits)[::-1][:3].tolist()
        score = 0.0
        for rank, pred in enumerate(top3):
            if pred == int(label):
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return float(np.mean(scores))

best_map3 = 0.0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.float()
        loss = map3_loss(logits, labels) / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    if len(train_loader) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.extend(outputs.logits.float().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_map3 = compute_map3(all_logits, all_labels)
    avg_loss = total_loss / max(1, len(train_loader))
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val MAP@3: {val_map3:.4f}")

    if val_map3 > best_map3:
        best_map3 = val_map3
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  Saved best model with MAP@3={best_map3:.4f}")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

"""

'os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"\n\nMODEL_NAME = "microsoft/deberta-v3-base"\nMAX_LEN = 192\nBATCH_SIZE = 2\nGRAD_ACCUM = 8\nEPOCHS = 17\nLR = 2e-5\nSAVE_PATH = "deberta_base_lora.pt"\nQUESTION_COL = "prompt2"\nANSWER_COL = "answer"\nPATIENCE = 4\n\ndf = pd.read_csv(TRAIN_PATH)\ndf["prompt2"] = df["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))\ndf.columns = [c.strip() for c in df.columns]\ntest_df_check = pd.read_csv(TEST_PATH)\ntest_df_check["prompt2"] = test_df_check["prompt"].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))\ntest_df_check.columns = [c.strip() for c in test_df_check.columns]\n\n\nOPTION_COLS = ["A", "B", "C", "D", "E"]\nLABEL_MAP = {v: i for i, v in enumerate(OPTION_COLS)}\n\ndf["label_idx"] = df[ANSWER_COL].str.strip().map(LABEL_MAP)\ndf = df.dropna(subset=["label_idx"])\ndf["label_idx"] = df["label_idx"].astype(int)\n\n\ntrain_df, val_df = train_test_split(df, test_size=0.15, ra

In [45]:
CHOICES    = list("ABCDE")
LABEL_MAP  = {v: i for i, v in enumerate(CHOICES)}

df = pd.read_csv(TRAIN_PATH)
df.columns = [c.strip() for c in df.columns]
df["label"] = df["answer"].str.strip().map(LABEL_MAP)
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)
print(f"Train rows: {len(df)}")


Train rows: 2000


In [46]:
def flatten_texts(df):
    texts = []
    for _, row in df.iterrows():
        q = str(row["prompt"])
        for c in CHOICES:
            texts.append(q + " [SEP] " + str(row[c]))
    return texts

def compute_map3(probs, labels):
    scores = []
    for p, l in zip(probs, labels):
        top3 = np.argsort(p)[::-1][:3].tolist()
        score = 0.0
        for rank, pred in enumerate(top3):
            if pred == int(l):
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return float(np.mean(scores))

def predict_probs(lr, X, n_rows):
    raw = lr.predict_proba(X)[:, 1]
    probs = raw.reshape(n_rows, 5)
    probs = np.exp(probs) / np.exp(probs).sum(axis=1, keepdims=True)
    return probs

In [47]:
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df["label"])

train_texts = flatten_texts(train_df)
val_texts   = flatten_texts(val_df)

train_labels_bin = []
for l in train_df["label"]:
    for i in range(5):
        train_labels_bin.append(1 if i == l else 0)

In [48]:
vec = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=1,
    analyzer="word"
)
X_train = vec.fit_transform(train_texts)
X_val   = vec.transform(val_texts)

In [49]:
lr = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1)
lr.fit(X_train, train_labels_bin)

LogisticRegression(max_iter=1000, n_jobs=-1)

In [50]:
val_probs = predict_probs(lr, X_val, len(val_df))
val_map3  = compute_map3(val_probs, val_df["label"].values)

print(f"Validation set map3 score {val_map3}")

Validation set map3 score 0.985


In [51]:
all_texts  = flatten_texts(df)
all_labels = []
for l in df["label"]:
    for i in range(5):
        all_labels.append(1 if i == l else 0)

vec_full = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=1,
    analyzer="word"
)

In [52]:
X_full = vec_full.fit_transform(all_texts)
lr_full = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1)
lr_full.fit(X_full, all_labels)

with open("tfidf_vec.pkl", "wb") as f:
    pickle.dump(vec_full, f)
with open("tfidf_lr.pkl", "wb") as f:
    pickle.dump(lr_full, f)

In [53]:
test_df = pd.read_csv(TEST_PATH)
test_df.columns = [c.strip() for c in test_df.columns]
print(f"Test rows: {len(test_df)}")

test_texts = flatten_texts(test_df)
X_test     = vec_full.transform(test_texts)
test_probs = predict_probs(lr_full, X_test, len(test_df))

predictions = []
for probs in test_probs:
    top3_idx = np.argsort(probs)[::-1][:3]
    predictions.append(" ".join([CHOICES[i] for i in top3_idx]))

id_col = test_df["id"] if "id" in test_df.columns else (test_df["ID"] if "ID" in test_df.columns else test_df.index)
submission = pd.DataFrame({"ID": id_col, "Prediction": predictions})
submission.to_csv("submission.csv", index=False)

print(submission.head(10))

Test rows: 500
   ID Prediction
0   1      A E C
1   2      B A C
2   3      B E D
3   4      E C A
4   5      C A B
5   6      D A B
6   7      E D C
7   8      B E A
8   9      C D E
9  10      B C D
